# Multi-Model Fair-RAG Notebook (LiquidAI LFM2.5-350M MLX 4-bit + SPLADE)

This notebook mirrors the T5-based setup, but switches the generator backend to MLX using `LiquidAI/LFM2.5-350M-MLX-4bit`.

Goals:
- keep existing flanT5Small results visible for comparison
- bootstrap the data/retrieval layout needed by the MLX generator label
- run the same Fair-RAG pipeline with BM25 first, then SPLADE later

In [ ]:
from pathlib import Path
from datetime import datetime
import importlib
import shutil
import subprocess
import time

import pandas as pd
from IPython.display import display

import notebook_experiment_utils as neu
importlib.reload(neu)

ExperimentConfig = neu.ExperimentConfig
find_repo_root = neu.find_repo_root
resolve_python_executable = neu.resolve_python_executable
ensure_paths_exist = neu.ensure_paths_exist
run_retriever_grid = neu.run_retriever_grid
run_deterministic_reference = neu.run_deterministic_reference
run_mmr_deterministic = neu.run_mmr_deterministic
run_gold_deterministic_reference = neu.run_gold_deterministic_reference
run_gold_baseline = neu.run_gold_baseline
normalize_eu_grid = neu.normalize_eu_grid
load_normalized_rows = neu.load_normalized_rows
load_raw_rows = neu.load_raw_rows
raw_results_path = neu.raw_results_path
add_ee_d_interval_bins = neu.add_ee_d_interval_bins
summarize_by_interval = neu.summarize_by_interval

In [ ]:
ROOT = find_repo_root()
PY = resolve_python_executable(ROOT)

profiles = {
    "weak": {"max_queries": 30, "n_samples": 10, "k": 5, "print_interval": 10},
    "balanced": {"max_queries": 833, "n_samples": 12, "k": 5, "print_interval": 40},
    "strong": {"max_queries": None, "n_samples": 100, "k": 5, "print_interval": 100},
}

profile_name = "weak"
profile = profiles[profile_name]

existing_generator = "flanT5Small"
target_generator = "lfm25MLX350M4bit"
target_model_repo = "LiquidAI/LFM2.5-350M-MLX-4bit"

cfg_existing = ExperimentConfig(
    root=ROOT,
    python_exe=PY,
    generator_name=existing_generator,
    lamp_num=4,
    retriever_name="bm25",
    alphas=(1, 2, 4, 8),
    max_queries=profile["max_queries"],
    n_samples=profile["n_samples"],
    k=profile["k"],
    remove_temp_files=True,
    skip_existing=True,
    mmr_base_retriever="bm25",
    mmr_lambda=0.6,
    run_tag=f"{profile_name}_{existing_generator}",
    print_interval=profile["print_interval"],
)

cfg_lfm_bm25 = ExperimentConfig(
    root=ROOT,
    python_exe=PY,
    generator_name=target_generator,
    lamp_num=4,
    retriever_name="bm25",
    alphas=(1, 2, 4, 8),
    max_queries=profile["max_queries"],
    n_samples=profile["n_samples"],
    k=profile["k"],
    remove_temp_files=True,
    skip_existing=True,
    mmr_base_retriever="bm25",
    mmr_lambda=0.6,
    run_tag=f"{profile_name}_{target_generator}_bm25",
    print_interval=profile["print_interval"],
)

cfg_lfm_splade = ExperimentConfig(
    root=ROOT,
    python_exe=PY,
    generator_name=target_generator,
    lamp_num=4,
    retriever_name="splade",
    alphas=(1, 2, 4, 8),
    max_queries=profile["max_queries"],
    n_samples=profile["n_samples"],
    k=profile["k"],
    remove_temp_files=True,
    skip_existing=True,
    mmr_base_retriever="splade",
    mmr_lambda=0.6,
    run_tag=f"{profile_name}_{target_generator}_splade",
    print_interval=profile["print_interval"],
)

ensure_paths_exist([cfg_existing.python_exe])

print("Root       :", ROOT)
print("Python     :", PY)
print("Profile    :", profile_name)
print("Existing   :", cfg_existing.generator_name, cfg_existing.retriever_name)
print("Target A   :", cfg_lfm_bm25.generator_name, cfg_lfm_bm25.retriever_name)
print("Target B   :", cfg_lfm_splade.generator_name, cfg_lfm_splade.retriever_name)
print("MLX model  :", target_model_repo)

## Bootstrap MLX Layout

The existing CLI expects generator-specific data and retrieval folders. This cell bootstraps them from the flanT5Small LaMP-4 assets so the MLX generator can run through the same pipeline.

In [ ]:
def bootstrap_mlx_layout(copy_bm25_retrieval: bool = True) -> None:
    source_data_dir = ROOT / "data" / f"lamp_utility_labels_{existing_generator}"
    target_data_dir = ROOT / "data" / f"lamp_utility_labels_{target_generator}"
    target_data_dir.mkdir(parents=True, exist_ok=True)

    shared_files = [
        f"{cfg_lfm_bm25.lamp_num}_user_dev_inputs.json",
        f"{cfg_lfm_bm25.lamp_num}_user_dev_outputs.json",
        f"{cfg_lfm_bm25.lamp_num}_relevance_mapping.tsv",
    ]

    for name in shared_files:
        src = source_data_dir / name
        dst = target_data_dir / name
        if not dst.exists():
            shutil.copy2(src, dst)
            print("copied", dst)

    if copy_bm25_retrieval:
        src_bm25 = ROOT / "retrieval" / "retrieval_results" / existing_generator / "bm25" / f"{cfg_lfm_bm25.lamp_num}.json"
        dst_bm25_dir = ROOT / "retrieval" / "retrieval_results" / target_generator / "bm25"
        dst_bm25_dir.mkdir(parents=True, exist_ok=True)
        dst_bm25 = dst_bm25_dir / src_bm25.name
        if not dst_bm25.exists():
            shutil.copy2(src_bm25, dst_bm25)
            print("copied", dst_bm25)

bootstrap_mlx_layout(copy_bm25_retrieval=True)

## MLX Backend Check

This verifies that `mlx` and `mlx_lm` import successfully before any experiment run.

In [ ]:
import mlx
import mlx_lm

print("mlx version   :", getattr(mlx, "__version__", "unknown"))
print("mlx_lm version:", getattr(mlx_lm, "__version__", "unknown"))
print("model repo     :", target_model_repo)

## Load Existing flanT5Small Results

Use this as the comparison anchor while building out the MLX runs.

In [ ]:
def _latest_notebook_output_dirs(cfg: ExperimentConfig) -> list[Path]:
    base = cfg.root / "experiment_results" / cfg.generator_name / f"lamp{cfg.lamp_num}" / cfg.retriever_name
    if not base.exists():
        return []
    out_dirs = [p for p in base.glob("notebook_outputs_*") if p.is_dir()]
    out_dirs.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return out_dirs

def _safe_read_csv(fp: Path) -> pd.DataFrame | None:
    if fp.exists():
        return pd.read_csv(fp)
    return None

def show_existing_outputs(cfg: ExperimentConfig, max_dirs: int = 2) -> None:
    dirs = _latest_notebook_output_dirs(cfg)[:max_dirs]
    if not dirs:
        print("No notebook_outputs_* directories found for", cfg.generator_name, cfg.retriever_name)
        return
    for out_dir in dirs:
        print(f"\n=== Existing outputs: {out_dir.name} ===")
        for name in [
            "ee_d_interval_summary.csv",
            "pvalues_vs_deterministic_by_interval.csv",
            "tableA_normalized_eu_by_disparity_interval_pooled.csv",
            "tableB_eu_diff_st_minus_det_by_disparity_interval_pooled.csv",
        ]:
            fp = out_dir / name
            df = _safe_read_csv(fp)
            if df is not None:
                print(f"\n{name} (rows={len(df)}):")
                display(df.head(20))

show_existing_outputs(cfg_existing, max_dirs=2)

## Pipeline Runner

This uses the same `experiment.py` / `notebook_experiment_utils.py` orchestration, but the generator backend now resolves to MLX via `generator/lm_mlx.py`.

In [ ]:
RUN_GOLD = True
RUN_RETRIEVER_GRID = False
RUN_DETERMINISTIC_REF = True
RUN_MMR_DETERMINISTIC = False
RUN_GOLD_DETERMINISTIC = False
RUN_NORMALIZE = False
RUN_INTERVAL_SUMMARY = False

DETERMINISTIC_ALPHA = 1

def run_stage(name: str, enabled: bool, fn):
    if not enabled:
        print(f"[skip] {name}")
        return
    print(f"\n=== START {name} @ {datetime.now().strftime('%H:%M:%S')} ===")
    t0 = time.time()
    fn()
    print(f"=== DONE {name} in {(time.time()-t0)/60:.2f} min ===")

def run_core_pipeline(cfg: ExperimentConfig) -> None:
    print("\nRunning config:", cfg)
    run_stage("gold baseline", RUN_GOLD, lambda: run_gold_baseline(cfg, alpha=8))
    run_stage(
        "deterministic reference",
        RUN_DETERMINISTIC_REF,
        lambda: run_deterministic_reference(cfg, alpha=DETERMINISTIC_ALPHA, output_suffix="_deterministic"),
    )
    run_stage(
        "deterministic mmr",
        RUN_MMR_DETERMINISTIC,
        lambda: run_mmr_deterministic(cfg, alpha=DETERMINISTIC_ALPHA, output_suffix="_mmr_deterministic"),
    )
    run_stage(
        "gold deterministic",
        RUN_GOLD_DETERMINISTIC,
        lambda: run_gold_deterministic_reference(cfg, alpha=DETERMINISTIC_ALPHA, output_suffix="_gold_deterministic"),
    )
    run_stage("retriever alpha grid", RUN_RETRIEVER_GRID, lambda: run_retriever_grid(cfg))
    run_stage("normalize alpha grid", RUN_NORMALIZE, lambda: normalize_eu_grid(cfg, normalization_scope="all-settings"))
    if RUN_INTERVAL_SUMMARY:
        def _interval():
            df = load_normalized_rows(cfg)
            df = add_ee_d_interval_bins(df)
            display(summarize_by_interval(df))
        run_stage("normalized interval summary", True, _interval)

## Run Targets

- Target A: LFM2.5-350M MLX 4-bit + BM25
- Target B: LFM2.5-350M MLX 4-bit + SPLADE

Start with BM25 on the weak profile, then expand to the alpha grid and SPLADE once the first run completes cleanly.

In [ ]:
RUN_TARGET_LFM_BM25 = False
RUN_TARGET_LFM_SPLADE = False

if RUN_TARGET_LFM_BM25:
    bootstrap_mlx_layout(copy_bm25_retrieval=True)
    run_core_pipeline(cfg_lfm_bm25)

if RUN_TARGET_LFM_SPLADE:
    bootstrap_mlx_layout(copy_bm25_retrieval=True)
    run_core_pipeline(cfg_lfm_splade)